# Retail Real-Time Recommendations: Feature Store + Model Training

This notebook sets up the Feature Store, trains two XGBoost models, and registers them.

**Prerequisites:** Run all setup SQL from the quickstart guide first.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql('USE DATABASE RETAIL_RECOMMENDATION_QS').collect()
session.sql('USE WAREHOUSE RETAIL_QS_WH').collect()
print(f'Connected: {session.get_current_database()}')


## 1. Feature Store Setup

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(session=session, database='RETAIL_RECOMMENDATION_QS',
    name='FEATURE_STORE', default_warehouse='RETAIL_QS_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
print('Feature Store initialized')


In [ ]:
customer_entity = Entity(name='CUSTOMER', join_keys=['CUSTOMER_ID'], desc='Retail customer')
fs.register_entity(customer_entity)
product_entity = Entity(name='PRODUCT', join_keys=['PRODUCT_ID'], desc='Product catalog')
fs.register_entity(product_entity)
print('Entities registered')


In [ ]:
customer_features_df = session.sql('''
    SELECT CUSTOMER_ID, AGE, TENURE_MONTHS,
        CASE LOYALTY_TIER WHEN 'Bronze' THEN 0 WHEN 'Silver' THEN 1
            WHEN 'Gold' THEN 2 WHEN 'Platinum' THEN 3 END AS LOYALTY_TIER_NUM,
        TOTAL_ORDERS, LIFETIME_SPEND, AVG_ORDER_VALUE, DAYS_SINCE_PURCHASE,
        CATEGORIES_PURCHASED, TOTAL_BROWSE_EVENTS, TOTAL_SESSIONS,
        AVG_BROWSED_PRICE, ADD_TO_CART_COUNT, CART_ABANDONMENT_RATE,
        STORE_RATIO, ENGAGEMENT_SCORE
    FROM RETAIL_RECOMMENDATION_QS.GOLD.CUSTOMER_360
''')

customer_fv = FeatureView(name='CUSTOMER_FEATURES', entities=[customer_entity],
    feature_df=customer_features_df, refresh_freq='30 minutes',
    desc='Customer 360 features for ML')
registered_customer_fv = fs.register_feature_view(feature_view=customer_fv, version='v1', block=True)
print(f'Customer features registered: {registered_customer_fv.name}')


In [ ]:
product_features_df = session.sql('''
    SELECT PRODUCT_ID, CATEGORY_ID, DEPARTMENT, PRICE, WEIGHT_LBS,
        CASE PRODUCT_TYPE WHEN 'daily_staple' THEN 0 WHEN 'mid_range' THEN 1
            WHEN 'premium' THEN 2 END AS PRODUCT_TIER,
        CASE WHEN IS_PERISHABLE THEN 1 ELSE 0 END AS IS_PERISHABLE
    FROM RETAIL_RECOMMENDATION_QS.RAW.PRODUCT_CATALOG
''')

product_fv = FeatureView(name='PRODUCT_FEATURES', entities=[product_entity],
    feature_df=product_features_df, refresh_freq='1 day',
    desc='Product catalog features for ML')
registered_product_fv = fs.register_feature_view(feature_view=product_fv, version='v1', block=True)
print(f'Product features registered: {registered_product_fv.name}')


## 2. Prepare Training Data

In [ ]:
training_events_df = session.sql('''
    SELECT e.CUSTOMER_ID, e.PRODUCT_ID, e.PRODUCT_CATEGORY AS CATEGORY_ID,
        CASE WHEN e.EVENT_TYPE IN ('order_complete', 'pos_purchase') THEN 1 ELSE 0 END AS PURCHASED
    FROM RETAIL_RECOMMENDATION_QS.RAW.EVENT_STREAM e
    WHERE e.EVENT_TYPE IN ('product_view', 'add_to_cart', 'order_complete', 'pos_purchase')
''')
print(f'Training events: {training_events_df.count()} rows')
training_events_df.group_by('PURCHASED').count().show()


In [ ]:
training_data = fs.retrieve_feature_values(
    spine_df=training_events_df,
    features=[registered_customer_fv, registered_product_fv])
training_pd = training_data.to_pandas()
print(f'Training data shape: {training_pd.shape}')


## 3. Train Propensity Model (Batch)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

propensity_features = [
    'AGE', 'TENURE_MONTHS', 'LOYALTY_TIER_NUM', 'TOTAL_ORDERS',
    'LIFETIME_SPEND', 'AVG_ORDER_VALUE', 'DAYS_SINCE_PURCHASE',
    'CATEGORIES_PURCHASED', 'TOTAL_BROWSE_EVENTS', 'TOTAL_SESSIONS',
    'CART_ABANDONMENT_RATE', 'ENGAGEMENT_SCORE'
]

training_pd = training_pd.loc[:, ~training_pd.columns.duplicated()]

propensity_df = training_pd.groupby(['CUSTOMER_ID', 'CATEGORY_ID'] + propensity_features).agg(
    PURCHASED=('PURCHASED', 'max')).reset_index()
X_prop = propensity_df[propensity_features].fillna(0)
y_prop = propensity_df['PURCHASED']
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_prop, y_prop, test_size=0.2, random_state=42, stratify=y_prop)

propensity_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    objective='binary:logistic', eval_metric='auc', random_state=42, n_jobs=-1)
propensity_model.fit(X_train_p, y_train_p)

y_pred_p = propensity_model.predict_proba(X_test_p)[:, 1]
auc_p = roc_auc_score(y_test_p, y_pred_p)
print(f'Propensity Model AUC: {auc_p:.4f}')


In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

reg = Registry(session=session, database_name='RETAIL_RECOMMENDATION_QS', schema_name='ML_REGISTRY')

propensity_mv = reg.log_model(
    model=propensity_model, model_name='CATEGORY_PROPENSITY_MODEL', version_name='v1',
    conda_dependencies=['scikit-learn', 'xgboost'],
    sample_input_data=X_train_p.head(10),
    target_platforms=['WAREHOUSE'],
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
    metrics={'auc': round(auc_p, 4)},
    comment='Customer category propensity model - batch daily')
print(f'Propensity model registered: {propensity_mv.model_name} v{propensity_mv.version_name}')


## 4. Train Ranking Model (Real-Time)

In [ ]:
ranking_features = [
    'AGE', 'TENURE_MONTHS', 'LOYALTY_TIER_NUM', 'TOTAL_ORDERS',
    'LIFETIME_SPEND', 'AVG_ORDER_VALUE', 'DAYS_SINCE_PURCHASE',
    'ENGAGEMENT_SCORE', 'AVG_BROWSED_PRICE',
    'PRICE', 'PRODUCT_TIER', 'IS_PERISHABLE'
]
ranking_df = training_pd.copy()
ranking_df['PRICE_RATIO'] = ranking_df['PRICE'] / ranking_df['AVG_BROWSED_PRICE'].clip(lower=1)
ranking_df['SPEND_MATCH'] = (ranking_df['PRICE'] / ranking_df['AVG_ORDER_VALUE'].clip(lower=1)).clip(upper=5)
all_ranking_features = ranking_features + ['PRICE_RATIO', 'SPEND_MATCH']

X_rank = ranking_df[all_ranking_features].fillna(0)
y_rank = ranking_df['PURCHASED']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rank, y_rank, test_size=0.2, random_state=42, stratify=y_rank)

ranking_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.08,
    objective='binary:logistic', eval_metric='auc',
    scale_pos_weight=len(y_train_r[y_train_r==0]) / max(len(y_train_r[y_train_r==1]), 1),
    random_state=42, n_jobs=-1)
ranking_model.fit(X_train_r, y_train_r)

y_pred_r = ranking_model.predict_proba(X_test_r)[:, 1]
auc_r = roc_auc_score(y_test_r, y_pred_r)
print(f'Ranking Model AUC: {auc_r:.4f}')


In [ ]:
ranking_mv = reg.log_model(
    model=ranking_model, model_name='PRODUCT_RANKING_MODEL', version_name='v1',
    conda_dependencies=['scikit-learn', 'xgboost'],
    sample_input_data=X_train_r.head(10),
    target_platforms=['SNOWPARK_CONTAINER_SERVICES'],
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
    metrics={'auc': round(auc_r, 4)},
    comment='Product ranking model - real-time SPCS inference')
print(f'Ranking model registered: {ranking_mv.model_name} v{ranking_mv.version_name}')


## 5. Deploy Ranking Model to SPCS

In [ ]:
ranking_mv.create_service(
    service_name='RANKING_MODEL_SERVICE',
    service_compute_pool='RETAIL_QS_COMPUTE_POOL',
    ingress_enabled=True)
print('Ranking model service deployment initiated.')


In [ ]:
import time
for _ in range(10):
    status = session.sql("CALL SYSTEM$GET_SERVICE_STATUS('RETAIL_RECOMMENDATION_QS.ML_REGISTRY.RANKING_MODEL_SERVICE')").collect()
    print(status)
    if 'READY' in str(status): break
    time.sleep(30)


In [ ]:
endpoints = session.sql('SHOW ENDPOINTS IN SERVICE RETAIL_RECOMMENDATION_QS.ML_REGISTRY.RANKING_MODEL_SERVICE').collect()
for row in endpoints:
    print(row.as_dict())


## 6. Verify

In [ ]:
models = session.sql('SHOW MODELS IN SCHEMA RETAIL_RECOMMENDATION_QS.ML_REGISTRY').collect()
for m in models:
    print(m.as_dict())
